In [1]:
# 1. ENVIRONMENT SETUP and Check Python and PySpark versions

import sys
import os
import pandas as pd
import pyspark

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml.fpm import FPGrowth

print("Python version:", sys.version)
print("PySpark version:", pyspark.__version__)

assert pyspark.__version__ == "3.5.9", (
    "Select the Python interpreter where PySpark 3.5.9 is installed."
)

Python version: 3.10.0 (tags/v3.10.0:b494f59, Oct  4 2021, 19:00:18) [MSC v.1929 64 bit (AMD64)]
PySpark version: 3.5.9


In [2]:
# 2. CREATE SPARK SESSION

spark = (
    SparkSession.builder
    .appName("EcommerceProductAssociationMining")
    .master("local[2]")
    .config("spark.ui.enabled", "false")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("Spark version:", spark.version)
spark.range(10).show()

Spark version: 3.5.9
+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
|  5|
|  6|
|  7|
|  8|
|  9|
+---+



In [3]:
# 3. LOAD THE DATASET

BASE_DIR = os.getcwd()
EXCEL_PATH = os.path.join(BASE_DIR, "Online Retail.xlsx")
CSV_PATH = os.path.join(BASE_DIR, "online_retail.csv")
OUTPUT_DIR = os.path.join(BASE_DIR, "output")

# Create output directory if it does not exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

if not os.path.exists(EXCEL_PATH):
    raise FileNotFoundError("Please place 'Online Retail.xlsx' in the project folder.")

# Convert Excel to CSV
print("Converting Excel to CSV...")
pd.read_excel(EXCEL_PATH, engine="openpyxl").to_csv(CSV_PATH, index=False)

# Load the CSV using PySpark
df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(CSV_PATH)
)

print("Dataset loaded successfully!")
print("Total rows:", df.count())
print("Total columns:", len(df.columns))

df.printSchema()
df.show(5, truncate=False)

Converting Excel to CSV...
Dataset loaded successfully!
Total rows: 541909
Total columns: 8
root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: double (nullable = true)
 |-- Country: string (nullable = true)

+---------+---------+-----------------------------------+--------+-------------------+---------+----------+--------------+
|InvoiceNo|StockCode|Description                        |Quantity|InvoiceDate        |UnitPrice|CustomerID|Country       |
+---------+---------+-----------------------------------+--------+-------------------+---------+----------+--------------+
|536365   |85123A   |WHITE HANGING HEART T-LIGHT HOLDER |6       |2010-12-01 08:26:00|2.55     |17850.0   |United Kingdom|
|536365   |71053    |WHITE METAL LANTERN                |6       |2010-12-01

In [4]:
# 4. DATA CLEANING

# Count original records
original_count = df.count()

# Clean the dataset
cleaned = (
    df
    .withColumn("InvoiceNo", F.trim(F.col("InvoiceNo")))
    .withColumn("StockCode", F.trim(F.col("StockCode")))
    .withColumn("Description", F.trim(F.col("Description")))
    .withColumn("Quantity", F.col("Quantity").cast("double"))
    .withColumn("UnitPrice", F.col("UnitPrice").cast("double"))

    # Remove rows with missing required values
    .dropna(subset=[
        "InvoiceNo",
        "StockCode",
        "Description",
        "Quantity",
        "UnitPrice"
    ])

    # Remove blank identifiers and descriptions
    .filter(F.length("InvoiceNo") > 0)
    .filter(F.length("StockCode") > 0)
    .filter(F.length("Description") > 0)

    # Remove cancelled invoices (invoice numbers beginning with C)
    .filter(~F.upper(F.col("InvoiceNo")).startswith("C"))

    # Remove returns and invalid prices
    .filter(F.col("Quantity") > 0)
    .filter(F.col("UnitPrice") > 0)

    # Keep product codes that contain only digits
    .filter(F.col("StockCode").rlike("^[0-9]+$"))

    # Remove duplicate rows and cache cleaned data
    .dropDuplicates()
    .cache()
)
# Trigger processing and count cleaned records
cleaned_count = cleaned.count()

# Display cleaning summary
print("DATA CLEANING SUMMARY")
print("-" * 35)
print(f"Original records : {original_count:,}")
print(f"Cleaned records  : {cleaned_count:,}")
print(f"Records removed  : {original_count - cleaned_count:,}")

print("\nDATASET STATISTICS")
print("-" * 35)
print("Distinct invoices :", cleaned.select("InvoiceNo").distinct().count())
print("Distinct products :", cleaned.select("StockCode").distinct().count())
print("Distinct countries:", cleaned.select("Country").distinct().count())

print("\nCleaned data preview:")
cleaned.show(5, truncate=False)

DATA CLEANING SUMMARY
-----------------------------------
Original records : 541,909
Cleaned records  : 472,253
Records removed  : 69,656

DATASET STATISTICS
-----------------------------------
Distinct invoices : 19549
Distinct products : 2868
Distinct countries: 38

Cleaned data preview:
+---------+---------+---------------------------------+--------+-------------------+---------+----------+--------------+
|InvoiceNo|StockCode|Description                      |Quantity|InvoiceDate        |UnitPrice|CustomerID|Country       |
+---------+---------+---------------------------------+--------+-------------------+---------+----------+--------------+
|536367   |22748    |POPPY'S PLAYHOUSE KITCHEN        |6.0     |2010-12-01 08:34:00|2.1      |13047.0   |United Kingdom|
|536367   |21777    |RECIPE BOX WITH METAL HEART      |4.0     |2010-12-01 08:34:00|7.95     |13047.0   |United Kingdom|
|536370   |10002    |INFLATABLE POLITICAL GLOBE       |48.0    |2010-12-01 08:45:00|0.85     |12583.0   

In [5]:
# 5. CREATE TRANSACTION BASKETS

baskets = (
    cleaned
    .groupBy("InvoiceNo")
    .agg(F.collect_set("StockCode").alias("items"))
    .filter(F.size("items") >= 2)
    .cache()
)

transaction_count = baskets.count()

print("Valid transactions:", transaction_count)
print("\nSample transaction baskets:")

baskets.show(5, truncate=False)

Valid transactions: 18052

Sample transaction baskets:
+---------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|InvoiceNo|items                                                                                                                                                                                                                                                                                                                                                                                                   |
+---------+----------------------------------------------------------------------------------------------------------------------------

In [6]:
# 6. APPLY THE FP-GROWTH ALGORITHM
# Configure the FP-Growth model
fp_growth = FPGrowth(
    itemsCol="items",
    minSupport=0.02,
    minConfidence=0.30
)

# Train the model
model = fp_growth.fit(baskets)

# Extract frequent itemsets
frequent_itemsets = model.freqItemsets.cache()

print("FP-Growth completed successfully!")
print("Frequent itemsets:", frequent_itemsets.count())

print("\nTop 20 frequent itemsets:")
frequent_itemsets.orderBy(
    F.desc("freq")
).show(20, truncate=False)

FP-Growth completed successfully!
Frequent itemsets: 430

Top 20 frequent itemsets:
+-------+----+
|items  |freq|
+-------+----+
|[22423]|1963|
|[47566]|1666|
|[20725]|1564|
|[84879]|1452|
|[22720]|1376|
|[22197]|1369|
|[21212]|1320|
|[22383]|1283|
|[20727]|1271|
|[22457]|1248|
|[23203]|1230|
|[22386]|1212|
|[22469]|1197|
|[21931]|1182|
|[22411]|1174|
|[22382]|1157|
|[22086]|1153|
|[22961]|1151|
|[20728]|1148|
|[23298]|1138|
+-------+----+
only showing top 20 rows



How to interpret the results
A rule such as {Product A} → {Product B} means:
- Antecedent: Product A, the item(s) already purchased.
- Consequent: Product B, the item(s) associated with that purchase.
- Confidence: How often Product B is purchased when Product A is purchased.
- Lift: How much more likely the purchase of Product B is when Product A is purchased, compared with its overall frequency. A lift above 1 indicates a positive association.

In [7]:
# 7. GENERATE ASSOCIATION RULES
# Extract association rules from the trained FP-Growth trained model
association_rules = model.associationRules.cache()

# Count generated rules
rule_count = association_rules.count()

print("ASSOCIATION RULES SUMMARY")
print("-" * 40)
print("Total association rules:", rule_count)

print("\nTop 20 rules by confidence:")
association_rules.orderBy(
    F.desc("confidence"),
    F.desc("lift")
).show(20, truncate=False)

ASSOCIATION RULES SUMMARY
----------------------------------------
Total association rules: 181

Top 20 rules by confidence:
+--------------+----------+------------------+------------------+--------------------+
|antecedent    |consequent|confidence        |lift              |support             |
+--------------+----------+------------------+------------------+--------------------+
|[22698, 22699]|[22697]   |0.9046822742474916|16.12174177168383 |0.02996897850653667 |
|[22698, 22697]|[22699]   |0.8560126582278481|14.55060311330425 |0.02996897850653667 |
|[22698]       |[22697]   |0.8337730870712401|14.85811625647584 |0.0350099711943275  |
|[22697, 22423]|[22699]   |0.8015717092337917|13.625209505732965|0.022601373808996232|
|[22698]       |[22699]   |0.7889182058047494|13.410123777012558|0.033126523376911146|
|[22699, 22423]|[22697]   |0.7771428571428571|13.848946551967282|0.022601373808996232|
|[22578]       |[22577]   |0.7592954990215264|25.477327785012257|0.021493463328163084|
|[226

Example interpretation of a discovered rule
The rule below was generated by the FP-Growth model:
- Antecedent (products already in the basket): Green Regency Teacup and Saucer + Roses Regency Teacup and Saucer
- Consequent (associated product): Pink Regency Teacup and Saucer
- Confidence: 70.53% — among transactions containing both antecedent products, about 70.53% also contained the consequent product.
- Lift: 16.80 — the consequent appears much more often with this antecedent combination than its overall transaction frequency would suggest.
These are associations in the transaction data, not proof that buying one product causes another to be purchased.

In [8]:
# 8. MAP PRODUCT CODES TO PRODUCT NAMES

# Create a product-code-to-name lookup
product_lookup = (
    cleaned
    .groupBy("StockCode")
    .agg(
        F.first("Description", ignorenulls=True).alias("ProductName")
    )
)
# Create a map containing product codes and names
product_map_df = product_lookup.agg(
    F.map_from_entries(
        F.collect_list(F.struct("StockCode", "ProductName"))
    ).alias("product_map")
)

# Convert antecedent and consequent product codes into readable product descriptions
readable_rules = (
    association_rules
    .crossJoin(product_map_df)
    .withColumn(
        "AntecedentProducts",
        F.expr("transform(antecedent, x -> element_at(product_map, x))")
    )
    .withColumn(
        "ConsequentProducts",
        F.expr("transform(consequent, x -> element_at(product_map, x))")
    )
    .select(
        "AntecedentProducts",
        "ConsequentProducts",
        "confidence",
        "lift",
        "support"
    )
)

# 9. EXPORT ASSOCIATION RULES
output_path = os.path.join(OUTPUT_DIR, "association_rules.csv")

export_df = readable_rules.toPandas()
export_df.to_csv(output_path, index=False)

print("Association rules exported successfully!")
print("File:", output_path)
print("Total rules exported:", len(export_df))


Association rules exported successfully!
File: c:\Users\HP\Desktop\Ecommerce_Product_Association_Spark\output\association_rules.csv
Total rules exported: 181


In [9]:
# 10. EXPORT FREQUENT ITEMSETS

itemsets_path = os.path.join(OUTPUT_DIR, "frequent_itemsets.csv")

itemsets_df = frequent_itemsets.toPandas()

# Convert lists of product codes into comma-separated strings
itemsets_df["items"] = itemsets_df["items"].apply(
    lambda items: ", ".join(map(str, items))
)

itemsets_df.to_csv(itemsets_path, index=False)

print("Frequent itemsets exported successfully!")
print("File:", itemsets_path)
print("Total frequent itemsets:", len(itemsets_df))

Frequent itemsets exported successfully!
File: c:\Users\HP\Desktop\Ecommerce_Product_Association_Spark\output\frequent_itemsets.csv
Total frequent itemsets: 430


In [10]:
# 11. FINAL PROJECT SUMMARY

print("=" * 50)
print("E-COMMERCE PRODUCT ASSOCIATION MINING")
print("=" * 50)

print(f"Original records       : {original_count:,}")
print(f"Cleaned records        : {cleaned_count:,}")
print(f"Records removed        : {original_count - cleaned_count:,}")
print(f"Valid transactions     : {transaction_count:,}")
print(f"Distinct products      : {cleaned.select('StockCode').distinct().count():,}")
print(f"Frequent itemsets      : {frequent_itemsets.count():,}")
print(f"Association rules      : {rule_count:,}")
print(f"Minimum support        : 0.02")
print(f"Minimum confidence     : 0.30")

print("\nOutput files:")
print("- output/association_rules.csv")
print("- output/frequent_itemsets.csv")

print("\nProject processing completed successfully!")

E-COMMERCE PRODUCT ASSOCIATION MINING
Original records       : 541,909
Cleaned records        : 472,253
Records removed        : 69,656
Valid transactions     : 18,052
Distinct products      : 2,868
Frequent itemsets      : 430
Association rules      : 181
Minimum support        : 0.02
Minimum confidence     : 0.30

Output files:
- output/association_rules.csv
- output/frequent_itemsets.csv

Project processing completed successfully!
